In [2]:
import kagglehub
import os
import shutil

import pandas as pd
# 1. Download dataset (returns cache path)
cache_path = kagglehub.dataset_download("prashantsingh001/recipes-dataset-64k-dishes")
print("Cache path:", cache_path)

# 2. Define target folder name (same as dataset name)
dataset_name = "recipes-dataset-64k-dishes"
target_path = os.path.join(os.getcwd(), dataset_name)

# 3. Copy the downloaded content to current directory
if not os.path.exists(target_path):
    shutil.copytree(cache_path, target_path)
    print("Copied to:", target_path)
else:
    print("Target folder already exists:", target_path)

C:\Users\tutul\PycharmProjects\LLMs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cache path: C:\Users\tutul\.cache\kagglehub\datasets\prashantsingh001\recipes-dataset-64k-dishes\versions\1
Target folder already exists: C:\Users\tutul\PycharmProjects\LLMs\recipes-dataset-64k-dishes


In [3]:
import os

# Show what's in your project directory
for root, dirs, files in os.walk("."):
    # skip virtualenv / cache noise
    if any(skip in root for skip in [".venv", "__pycache__", ".git", "kagglehub"]):
        continue
    for f in files:
        if f.endswith((".csv", ".json")):
            print(os.path.join(root, f))

.\.mcp.json
.\.claude\settings.local.json
.\recipes-dataset-64k-dishes\1_Recipe_csv.csv
.\recipes-dataset-64k-dishes\2_Recipe_json.json


In [4]:
import pandas as pd

csv_path = r".\recipes-dataset-64k-dishes\1_Recipe_csv.csv"
df = pd.read_csv(csv_path)

print(df.shape)
print(df.columns.tolist())
df

(62126, 8)
['recipe_title', 'category', 'subcategory', 'description', 'ingredients', 'directions', 'num_ingredients', 'num_steps']


,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps
0,Air Fryer Potato Slices with Dipping Sauce,Air Fryer Recipes,Air Fryer Recipes,"These air fryer potato slices, served with a b...","[""3/4 cup ketchup"", ""1/2 cup beer"", ""1 tablesp...","[""Combine ketchup, beer, Worcestershire sauce,...",9,5
1,Gochujang Pork Belly Bites,Air Fryer Recipes,Air Fryer Recipes,These gochujang pork belly bites are sweet and...,"[""1 pound pork belly"", ""1/4 cup gochujang"", ""2...","[""Preheat an air fryer to 400 degrees F (200 d...",5,4
2,3-Ingredient Air Fryer Everything Bagel Chicke...,Air Fryer Recipes,Air Fryer Recipes,These 3-ingredient air fryer everything bagel ...,"[""1 \u00bc pounds chicken tenders"", ""1 tablesp...","[""Gather all ingredients. Preheat an air fryer...",3,4
3,Air Fryer Everything Bagel Chicken Cutlets,Air Fryer Recipes,Air Fryer Recipes,These air fryer everything bagel chicken cutle...,"[""4 chicken cutlets (about 1 pound total)"", ""s...","[""Preheat an air fryer to 400 degrees F (200 d...",9,9
4,Air Fryer Honey Sriracha Salmon Bites,Air Fryer Recipes,Air Fryer Recipes,These air fryer honey Sriracha salmon bites ar...,"[""1 tablespoon soy sauce"", ""1 tablespoon honey...","[""Preheat an air fryer to 400 degrees F (200 d...",5,5
...,...,...,...,...,...,...,...,...
62121,Rice Zucchini Muffins,Zucchini Breads,Zucchini Muffins,I created this for my son who has many food al...,"[""1 cup rice flour"", ""1 cup cooked rice"", ""\u0...","[""Preheat oven to 350 degrees F (175 degrees C...",11,3
62122,Gluten-Free Zucchini Muffins,Zucchini Breads,Zucchini Muffins,These gluten-free muffins are so good you won'...,"[""cooking spray"", ""1 cup vanilla whey protein ...","[""Preheat the oven to 350 degrees F (175 degre...",15,5
62123,Five Spice Muffins,Zucchini Breads,Zucchini Muffins,These muffins are versatile and always moist a...,"[""2 cups shredded zucchini"", ""\u2153 cup boili...","[""Preheat oven to 350 degrees F (175 degrees C...",14,5
62124,Low-Fat Chocolate Zucchini Mini Muffins,Zucchini Breads,Zucchini Muffins,A sweet snack your kids will gobble up and you...,"[""cooking spray"", ""2 (1 ounce) squares unsweet...","[""Preheat oven to 350 degrees F (175 degrees C...",14,5


## Cleaning & deduplication
Filters chosen from a profile of the raw data: counts match list lengths, no mojibake, 60% of rows are cross-category duplicates, and 1-2 ingredient rows are mostly truncated lists (e.g. only `cooking spray`).

In [5]:
import json, re

raw = pd.read_csv(csv_path)
clean = raw.copy()
clean["ingredients"] = clean["ingredients"].map(json.loads)
clean["directions"] = clean["directions"].map(json.loads)
log = [("raw", len(clean))]

def step(name, mask):
    """Keep rows where mask is True and record the remaining row count."""
    global clean
    removed = (~mask).sum()
    clean = clean[mask].copy()
    log.append((f"{name} (-{removed})", len(clean)))

# 1. Truncated / unusable ingredient lists break ingredient-compliance scoring
step("ingredients >= 4", clean["ingredients"].map(len) >= 4)
# 2. Stub directions
step("directions >= 60 chars", clean["directions"].map(lambda d: len(" ".join(d))) >= 60)
# 3. Non-human-food categories (cocktails are kept: valid for culinary tests)
non_food = ["Pet Food", "Diy Recipes"]
step("drop Pet Food / DIY", ~(clean["category"].isin(non_food) | clean["subcategory"].isin(non_food)))

# Single-step recipes are usable, so flag instead of dropping
clean["single_step"] = clean["directions"].map(len) < 2

In [6]:
def norm_text(s):
    return re.sub(r"[^a-z0-9 ]+", "", s.lower()).strip()

clean["title_norm"] = clean["recipe_title"].map(norm_text)
clean["ing_key"] = clean["ingredients"].map(lambda l: "|".join(sorted(norm_text(x) for x in l)))

# Same recipe is listed under several categories: keep the most specific label.
# "specific" = subcategory differs from category, then the rarest subcategory.
sub_freq = clean["subcategory"].map(clean["subcategory"].value_counts())
clean["_generic"] = (clean["subcategory"] == clean["category"]).astype(int)
clean["_sub_freq"] = sub_freq

# keep the categories each recipe appeared under before collapsing
all_cats = (clean.groupby(["title_norm", "ing_key"])["category"]
            .agg(lambda s: sorted(set(s))).rename("all_categories"))

golden_pool = (clean.sort_values(["_generic", "_sub_freq"])
               .drop_duplicates(["title_norm", "ing_key"], keep="first")
               .merge(all_cats, on=["title_norm", "ing_key"])
               .drop(columns=["_generic", "_sub_freq"])
               .reset_index(drop=True))
log.append(("dedup (title+ingredients)", len(golden_pool)))

pd.DataFrame(log, columns=["step", "rows"])

,step,rows
0,raw,62126
1,ingredients >= 3 (-1090),61036
2,directions >= 60 chars (-6),61030
3,drop Pet Food / DIY (-112),60918
4,dedup (title+ingredients),24547


In [7]:
# Sanity checks
assert not golden_pool.duplicated(["title_norm", "ing_key"]).any()
assert golden_pool["ingredients"].map(len).min() >= 3
print(golden_pool.shape, "| single-step:", golden_pool["single_step"].sum())
print(golden_pool["category"].value_counts().head(10))

(24547, 12) | single-step: 1688
category
Appetizers And Snacks    1228
Cakes                    1112
Main Dishes               978
Healthy Recipes           685
Breads                    641
Breakfast And Brunch      600
Beef Recipes              596
Cookies                   508
Pies                      489
Mexican                   483
Name: count, dtype: int64


In [9]:
df[df.recipe_title.str.contains("Bob's Smoked Pulled Pork")]

,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps
47363,Bob's Smoked Pulled Pork,Main Dishes,Pork Main Dishes,This smoked pulled pork recipe makes the most ...,"[""1 (8 pound) pork shoulder roast"", ""1 quart a...","[""Gather all ingredients."", ""Place pork should...",2,7
58775,Bob's Smoked Pulled Pork,Pork,Pork Shoulder,This smoked pulled pork recipe makes the most ...,"[""1 (8 pound) pork shoulder roast"", ""1 quart a...","[""Gather all ingredients."", ""Place pork should...",2,7
59032,Bob's Smoked Pulled Pork,Pork,Bbq & Grilled Pork,This smoked pulled pork recipe makes the most ...,"[""1 (8 pound) pork shoulder roast"", ""1 quart a...","[""Gather all ingredients."", ""Place pork should...",2,7
59257,Bob's Smoked Pulled Pork,Pork,Pork Main Dishes,This smoked pulled pork recipe makes the most ...,"[""1 (8 pound) pork shoulder roast"", ""1 quart a...","[""Gather all ingredients."", ""Place pork should...",2,7


## Add xiaoxl recipes that have verified nutrition
`data/xiaoxl_nutrition_checked.csv` comes from `scripts/pull_xiaoxl.py` (download + keep rows with nutrition), a dedup against the Kaggle set, and `scripts/verify_xiaoxl.py` (recompute macros from ingredients with USDA data via `recipe_eval/nutrition.py`). `macro_verified` = our math and the label agree (calories within 25%, other macros within 40%, ingredient coverage >= 80%). Nutrition values are per serving and are only as accurate as that agreement (median error ~15%).

In [ ]:
import ast

x = pd.read_csv("data/xiaoxl_nutrition_checked.csv")
xv = x[x["macro_verified"]].copy()

def split_ingredients(s):
    return [i.strip() for i in str(s).split(" ; ") if i.strip()]

def parse_steps(s):
    try:
        return list(ast.literal_eval(s))
    except Exception:
        return [str(s)]

xiaoxl_std = pd.DataFrame({
    "recipe_title": xv["title"],
    "category": xv["category"],
    "subcategory": xv["category"],
    "description": xv["description"],
    "ingredients": xv["ingredients"].map(split_ingredients),
    "directions": xv["instructions_list"].map(parse_steps),
    "servings": xv["servings"],
    "kcal": xv["calories"],
    "protein_g": xv["protein_g"],
    "carb_g": xv["carbohydrates_g"],
    "fat_g": xv["fat_g"],
    "fiber_g": xv["dietary_fiber_g"],
    "sodium_mg": xv["sodium_mg"],
    "source": "xiaoxl",
    "nutrition_basis": "label, agrees with USDA recompute",
}).reset_index(drop=True)

# same quality rules as the Kaggle rows
xiaoxl_std = xiaoxl_std[xiaoxl_std["ingredients"].map(len) >= 4]
xiaoxl_std = xiaoxl_std[xiaoxl_std["directions"].map(lambda d: len(" ".join(d))) >= 60]
xiaoxl_std["single_step"] = xiaoxl_std["directions"].map(len) < 2

kaggle_std = golden_pool[["recipe_title", "category", "subcategory", "description",
                          "ingredients", "directions", "single_step"]].copy()
kaggle_std["source"] = "kaggle"
kaggle_std["nutrition_basis"] = "none"

merged = pd.concat([kaggle_std, xiaoxl_std], ignore_index=True)
merged.insert(0, "recipe_id", [f"r{i:05d}" for i in range(len(merged))])
merged["has_nutrition"] = merged["kcal"].notna()
merged.groupby("source").agg(recipes=("recipe_id", "size"), with_nutrition=("has_nutrition", "sum"))

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
merged.to_json("data/recipes_merged.jsonl", orient="records", lines=True, force_ascii=False)

# Sanity checks
assert merged["recipe_id"].is_unique
assert merged["ingredients"].map(len).min() >= 4
assert merged.loc[merged["has_nutrition"], "servings"].gt(0).all()
print(merged.shape, "| saved data/recipes_merged.jsonl")
print(merged["single_step"].sum(), "single-step recipes flagged")

In [ ]:
## Computed nutrition for recipes without servings
This step also applies a **calorie cap**: recipes whose whole-recipe total is above 4,000 kcal (big-batch recipes) are dropped, and the remaining Kaggle recipes get an estimated `servings = round(total / 750 kcal)` (flag `servings_estimated`). Kaggle recipes have no recorded servings, so true per-serving nutrition cannot be recovered reliably (text rarely states servings; guessing them from calories was right within 25% only about 20-27% of the time; a model-based estimate was right within 25% about half the time). What *can* be computed from the ingredients with USDA data are whole-recipe **totals**, and therefore **shares of calories** (protein/carb/fat), which do not depend on servings. `scripts/add_computed_nutrition.py` adds them for recipes with at least 80% ingredient coverage and records the source in `nutrition_basis`.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/add_computed_nutrition.py"], check=True)
merged = pd.read_json("data/recipes_merged.jsonl", lines=True)
merged["nutrition_basis"].value_counts().to_frame("recipes")

## Preference labels, evaluation examples and scoring
Code lives in `recipe_eval/` (`labels.py`, `scoring.py`, `nutrition.py`). Thresholds are in `labels.THRESHOLDS`.

- **Macro tags** (`high_protein`, `high_carb`, `low_carb`, `keto`) are defined on **shares of calories**, so they need no servings and work for every recipe with real or computed nutrition. Computed (Kaggle) shares must clear a threshold by 3 points (`COMPUTED_MARGIN`). `low_calorie` needs real servings, so it exists only for the xiaoxl rows.
- **Keyword tags** (`vegetarian`, `vegan`, `dairy_free`, `gluten_free`, `spicy`) are read from the ingredient list (approximate).
- Each example = ingredients the user has (+ 0-2 distractors that need not be used) + 1-2 mood tags; the golden recipe is the reference.

In [ ]:
from recipe_eval import labels as L

merged["tags"] = [L.row_tags(r) for r in merged.itertuples()]
tag_counts = pd.Series([t for ts in merged["tags"] for t in ts]).value_counts()
tag_counts.to_frame("recipes")

In [ ]:
import subprocess, sys

# per_tag examples per mood tag (10 tags) -> data/eval_examples.jsonl
subprocess.run([sys.executable, "scripts/build_examples.py", "120", "7"], check=True)
examples = pd.read_json("data/eval_examples.jsonl", lines=True)
examples[["example_id", "source", "request"]].head()

### Scoring a model answer
`score_response(request, response, calc)` returns three scores in [0, 1] and an overall score:
- **compliance** (hard gate): a recipe that uses none of the available ingredients gives overall 0; extra ingredients outside the list are allowed (staples like salt, pepper, water, oil are always allowed)
- **preference**: how well the recipe meets each requested tag (macro tags use USDA-computed nutrition with a 15% tolerance)
- **feasibility**: rule checks (steps present, ingredients used in steps, sane temperatures/times, amounts given); optionally mix in a 1-5 LLM-judge rating

`overall = 0.3*compliance + 0.4*preference + 0.3*feasibility`. Response format: `{'servings': n, 'ingredients': [lines with amounts], 'steps': [str]}`.

In [ ]:
from recipe_eval import scoring as S
from recipe_eval.nutrition import Calculator

calc = Calculator()
ex = examples.iloc[0]
ref = ex["golden"].get("servings")

# Sanity check: the golden recipe scored as if it were the model's answer should be ~1.0
good = S.score_response(ex["request"], ex["golden"], calc, reference_servings=ref)

# Extra ingredients are allowed: adding one the user does not have still scores fine
extra_answer = dict(ex["golden"])
extra_answer["ingredients"] = list(ex["golden"]["ingredients"]) + ["2 cups truffle oil"]
extra = S.score_response(ex["request"], extra_answer, calc, reference_servings=ref)

# A bad answer: uses NONE of the user's ingredients -> compliance hard-fails -> overall 0
bad_answer = dict(ex["golden"])
bad_answer["ingredients"] = ["2 cups truffle oil", "1 lb wagyu beef", "1 cup saffron rice"]
bad = S.score_response(ex["request"], bad_answer, calc, reference_servings=ref)

print("golden answer :", good["overall"])
print("with an extra :", extra["overall"], "| extras:", extra["compliance"]["extras"])
print("no user item  :", bad["overall"], "| hard fail:", bad["compliance"]["hard_fail"])

In [ ]:
# Golden recipes as answers: expect a mean near 1 and no false compliance failures
scores = [S.score_response(r["request"], r["golden"], calc, reference_servings=r["golden"].get("servings"))["overall"] for _, r in examples.sample(200, random_state=0).iterrows()]
print("mean overall:", round(sum(scores) / len(scores), 3), "| min:", min(scores))